Part 1: Experiment Tracking with MLflow

Task 1.1: MLflow Setup & Experiment Logging

In [ ]:
#python -m venv venv
#venv\Scripts\activate.bat
#pip install -r requirements.txt
#python -m mlflow ui --port 5000
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import os
import requests
import polars as pl
import duckdb as db


def dataset_dowload():
    try:
        file = "yellow_taxi_data.parquet"
        if not os.path.exists(file):
            print("the taxi data file is missing,attempting to download")
            local_filename = "yellow_taxi_data.parquet"
            url1 ="https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
            with requests.get(url1, stream=True) as r:
                r.raise_for_status()
                with open(local_filename, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
            print("yelllow completed")
            df = pl.read_parquet(file)
    except Exception as e:
     print("this error occured downloading the parquet file: ", e)

    try:
        local_filename2 = "taxi_zone_data.csv"
        if not os.path.exists(local_filename2):
            url2 ="https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
            with requests.get(url2, stream=True) as r:
                r.raise_for_status()
                with open(local_filename2, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
            print("zone completed")
            df2 = pl.read_csv(local_filename2)
    except Exception as e:
        print("\n the following error occured with downloading the csv file: ", e)
    return df,df2

def dataset_cleaning(df, csv_zones):
    df = df.filter(pl.col("payment_type") ==1)

    df = df.with_columns([
        ((pl.col("tpep_dropoff_datetime") -pl.col("tpep_pickup_datetime"))
         .dt.total_minutes()).alias("trip_duration_minutes"),
        (pl.col("trip_distance")/((pl.col("tpep_dropoff_datetime") -pl.col("tpep_pickup_datetime"))
         .dt.total_minutes()/60)).alias("trip_speed_mph"),
        
        pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),
        pl.col("tpep_pickup_datetime").dt.weekday().alias("pickup_day_of_week"),
    ])

    df = df.with_columns([
        pl.col("trip_distance").log().alias("log_trip_distance"),
        (pl.col("fare_amount") /pl.col("trip_distance")).alias("fare_per_mile"),
        (pl.col("fare_amount") /pl.col("trip_duration_minutes")).alias("fare_per_minute")
    ])

    df = df.join(csv_zones.select(["LocationID","Borough"]), 
                 left_on="PULocationID",right_on="LocationID",how="left").rename({"Borough":"pickup_Borough"})
    
    df = df.join(csv_zones.select(["LocationID", "Borough"]), 
                 left_on="DOLocationID",right_on="LocationID",how="left").rename({"Borough":"dropoff_Borough"})

    X =df.to_pandas()
    y= X['tip_amount']

    for prefix in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
        X[f"{prefix}_hour"] =X[prefix].dt.hour
        X[f"{prefix}_day"] =X[prefix].dt.day
        X[f"{prefix}_month"]= X[prefix].dt.month


    for col in ['pickup_Borough', 'dropoff_Borough']:
        X[col] = X[col].str.replace(' ', ' ').str.replace('/', '/').str.replace('N_A', 'N_A')
    
    categorical_cols =['store_and_fwd_flag','pickup_Borough','dropoff_Borough']
    X =pd.get_dummies(X,columns=categorical_cols,drop_first=False)

    required_features = [
        'VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type',
        'fare_amount', 'extra', 'mta_tax', 'tolls_amount', 'improvement_surcharge', 'congestion_surcharge', 'Airport_fee',
        'trip_duration_minutes', 'trip_speed_mph', 'pickup_hour', 'pickup_day_of_week', 'log_trip_distance',
        'fare_per_mile', 'fare_per_minute', 'tpep_pickup_datetime_hour', 'tpep_pickup_datetime_day', 'tpep_pickup_datetime_month','tpep_dropoff_datetime_hour', 'tpep_dropoff_datetime_day', 'tpep_dropoff_datetime_month', 'store_and_fwd_flag_Y',
        'pickup_Borough_Brooklyn', 'pickup_Borough_EWR', 'pickup_Borough_Manhattan', 'pickup_Borough_N_A', 'pickup_Borough_Queens', 'pickup_Borough_Staten_Island', 'pickup_Borough_Unknown',
        'dropoff_Borough_Brooklyn', 'dropoff_Borough_EWR', 'dropoff_Borough_Manhattan', 'dropoff_Borough_N_A', 'dropoff_Borough_Queens', 'dropoff_Borough_Staten_Island', 'dropoff_Borough_Unknown'
    ]

    # 9. Alignment Logic
    # Add missing columns with 0 (e.g., if EWR isn't in this batch)
    for col in required_features:
        if col not in X.columns:
            X[col] = 0
            
    # Select only the 42 features (this automatically drops Bronx and raw datetimes)
    X = X[required_features]

    # 10. Final cleanup
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    bool_cols = X.select_dtypes(include=['bool']).columns
    X[bool_cols] = X[bool_cols].astype(int)

    return X, y


def dataset_split(X,y):
    return train_test_split(X, y,test_size=0.2,random_state=42)

def load_clean_dataset():
    CLEANED_FILE ="cleaned_taxi___data.parquet"

    REQUIRED_COLUMNS = [
        "trip_duration_minutes", "trip_speed_mph",
        "pickup_hour","pickup_day_of_week",
        "log_trip_distance", "fare_per_mile","fare_per_minute",
        "tpep_pickup_datetime_hour","tpep_pickup_datetime_day","tpep_pickup_datetime_month",
        "tpep_dropoff_datetime_hour","tpep_dropoff_datetime_day", "tpep_dropoff_datetime_month"
    ]

    if os.path.exists(CLEANED_FILE):
        print("cleaned dataset loaded")
        df = pd.read_parquet(CLEANED_FILE)
        datetime_cols =df.select_dtypes(include=["datetime64[ns]"]).columns#fast api cannot handle datetime datatypes

        for col in datetime_cols:
            df[f"{col}_hour"] =df[col].dt.hour
            df[f"{col}_day"] =df[col].dt.day
            df[f"{col}_month"] =df[col].dt.month
            df[col] =df[col].astype("int64") //10**9
    
        if "trip_speed_mph" not in df.columns and "trip_distance" in df.columns:
            df["trip_speed_mph"] = df["trip_distance"] /(df["trip_duration_minutes"] /60)
        if "log_trip_distance" not in df.columns:
            df["log_trip_distance"] =np.log1p(df["trip_distance"])
        if "fare_per_mile" not in df.columns:
            df["fare_per_mile"] =df["fare_amount"] /df["trip_distance"]
        if "fare_per_minute" not in df.columns:
            df["fare_per_minute"] =df["fare_amount"] /df["trip_duration_minutes"]
        for col in REQUIRED_COLUMNS:
            if col not in df.columns:
                df[col] =0
        df =df.replace([np.inf,-np.inf],np.nan).fillna(0)

        return df

    print("cleaned dataset not found, making it")
    df, zones =dataset_dowload()
    X, y =dataset_cleaning(df, zones)

    df_full = X.copy()
    df_full["tip_amount"] =y

    df_full.to_parquet(CLEANED_FILE)
    print("cleaned dataset created and saved.")

    return df_full

def train_and_evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds =model.predict(X_test)

    return {
        "model":model,
        "MAE":mean_absolute_error(y_test, preds),
        "RMSE":np.sqrt(mean_squared_error(y_test, preds)),
        "R2":r2_score(y_test, preds)}

def log_metrics(y_true, y_pred):
    mlflow.log_metric("MAE",mean_absolute_error(y_true, y_pred))
    mlflow.log_metric("RMSE", np.sqrt(mean_squared_error(y_true, y_pred)))
    mlflow.log_metric("R2",r2_score(y_true, y_pred))




Set up a local MLflow tracking server and configure your notebook to log to it

In [38]:
df= load_clean_dataset()
X = df.drop(columns=["tip_amount"])
print(X.dtypes)
y =df["tip_amount"]
X_train, X_test,y_train,y_test =dataset_split(X, y)
mlflow.set_tracking_uri("http://localhost:5000") # code if connection is refused


cleaned dataset not found, making it
the taxi data file is missing,attempting to download
yelllow completed
zone completed
cleaned dataset created and saved.
VendorID                           int32
passenger_count                    int64
trip_distance                    float64
RatecodeID                         int64
PULocationID                       int32
DOLocationID                       int32
payment_type                       int64
fare_amount                      float64
extra                            float64
mta_tax                          float64
tolls_amount                     float64
improvement_surcharge            float64
congestion_surcharge             float64
Airport_fee                      float64
trip_duration_minutes              int64
trip_speed_mph                   float64
pickup_hour                         int8
pickup_day_of_week                  int8
log_trip_distance                float64
fare_per_mile                    float64
fare_per_minute       

Display a screenshot or output of the MLflow UI showing all logged runs<img src="logged runs.png">


Create an MLflow experiment named "taxi-tip-prediction"
• Re-train (or load and evaluate) at least 2 models from Assignment 2 (e.g., Random
Forest and your tuned or neural network model),

In [39]:
mlflow.set_experiment("taxi-tip-prediction")

def run_experiment(model,model_name,params,X_train,X_test,y_train,y_test):
    with mlflow.start_run(run_name=model_name):
        if params:
            mlflow.log_params(params)
        results =train_and_evaluate(model,X_train,X_test,y_train,y_test)

        mlflow.log_metrics({
            "MAE":results["MAE"],
            "RMSE":results["RMSE"],
            "R2":results["R2"]
        })
        mlflow.set_tag("model_type",model_name)
        mlflow.set_tag("dataset","NYC Taxi")
        mlflow.sklearn.log_model(results["model"],"model")
        print(model_name," training completed")
        return results

# random forest
rf_params ={
    "n_estimators":100,
    "max_depth":10,
    "random_state":42
}
rf_results =run_experiment(RandomForestRegressor(**rf_params),"randomforest",rf_params,X_train,X_test,y_train,y_test)

# linear regression

lr_results = run_experiment(LinearRegression(),"linear regression",{},X_train,X_test,y_train,y_test)

2026/04/17 22:55:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 22:55:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


randomforest  training completed
🏃 View run randomforest at: http://localhost:5000/#/experiments/1/runs/ee4a67872851455e88732d8d9ce922f5
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/04/17 22:56:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 22:56:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


linear regression  training completed
🏃 View run linear regression at: http://localhost:5000/#/experiments/1/runs/1ba786630e6e40dab7e5e63767404f8a
🧪 View experiment at: http://localhost:5000/#/experiments/1


Use the MLflow UI or API to compare all logged runs side-by-side; include a screenshot
showing the comparison view


linear regression results
<img src="3610 lr results.png">
random forest results
<img src="3610 random forest results.png">


Write 2-3 sentences explaining which model performed best and why, referencing the
logged metrics

the random forest model preformed better with lower errors in both MAE and rmse with a higher r2 value. with a lower mae of 1.18 compared to 1.20 and rmse of 2.2 vs 2.3 of the linear regressor model and r2 of 0.64 compared to 0.63. making the randome ofest an overall better predictor

In [40]:
experiment =mlflow.get_experiment_by_name("taxi-tip-prediction")
runs =mlflow.search_runs(experiment_ids=[experiment.experiment_id])
print(runs)

                             run_id experiment_id    status  \
0  1ba786630e6e40dab7e5e63767404f8a             1  FINISHED   
1  ee4a67872851455e88732d8d9ce922f5             1  FINISHED   
2  618895cdccb740b8b997f3be2ca1aa7c             1  FINISHED   
3  3db3207db2f84173be5e7496e643ae0f             1  FINISHED   

                                        artifact_uri  \
0  mlflow-artifacts:/1/1ba786630e6e40dab7e5e63767...   
1  mlflow-artifacts:/1/ee4a67872851455e88732d8d9c...   
2  mlflow-artifacts:/1/618895cdccb740b8b997f3be2c...   
3  mlflow-artifacts:/1/3db3207db2f84173be5e7496e6...   

                        start_time                         end_time  \
0 2026-04-18 02:56:07.672000+00:00 2026-04-18 02:56:24.099000+00:00   
1 2026-04-18 02:30:39.430000+00:00 2026-04-18 02:56:07.585000+00:00   
2 2026-04-16 19:08:44.070000+00:00 2026-04-16 19:09:01.859000+00:00   
3 2026-04-16 18:47:24.175000+00:00 2026-04-16 19:08:44.015000+00:00   

   metrics.MAE  metrics.R2  metrics.RMSE params

Register your best-performing model in the MLflow Model Registry with a descriptive
model name (e.g., "taxi-tip-regressor") and a version description documenting its
performance

In [41]:
run_id = "one"
model_name = "taxi-tip-regressor"

mlflow.register_model(
    model_uri="runs:/3db3207db2f84173be5e7496e643ae0f/model",
    name=model_name
)

client = MlflowClient()
client.transition_model_version_stage(
 name="taxi-tip-regressor",
 version=1,
 stage="Production"
)
client.update_model_version(
 name="taxi-tip-regressor",
 version=1,
 description="Random Forest with MAE=1.185, R2=0.64. Trained on april 2026."
)

Registered model 'taxi-tip-regressor' already exists. Creating a new version of this model...
2026/04/17 22:56:24 WARNING mlflow.tracking._model_registry.fluent: Run with id 3db3207db2f84173be5e7496e643ae0f has no artifacts at artifact path 'model', registering model based on models:/m-905fe4045f22487a89c0ae6279bda9be instead
2026/04/17 22:56:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: taxi-tip-regressor, version 7
Created version '7' of model 'taxi-tip-regressor'.
C:\Users\shiva\AppData\Local\Temp\ipykernel_11776\26493900.py:10: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_sta

<ModelVersion: aliases=[], creation_timestamp=1776368717818, current_stage='Production', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='Random Forest with MAE=1.185, R2=0.64. Trained on april 2026.', last_updated_timestamp=1776480985050, metrics=None, model_id=None, name='taxi-tip-regressor', params=None, run_id='3db3207db2f84173be5e7496e643ae0f', run_link='', source='models:/m-905fe4045f22487a89c0ae6279bda9be', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

Demonstrate loading the model from the registry and making a sample prediction

In [ ]:
model = mlflow.sklearn.load_model(f"models:/{model_name}/Production")
sample_input = X_test[:1].copy() # using sample from test set

# the naming was not matching, and the deadline was to close to retrain to a cheap, non porduction fix was used,
rename_map = {
    'dropoff_Borough_N_A':'dropoff_Borough_N/A',
    'dropoff_Borough_Staten_Island':'dropoff_Borough_Staten Island',
    'pickup_Borough_N_A':'pickup_Borough_N/A',
    'pickup_Borough_Staten_Island':'pickup_Borough_Staten Island'
}
sample_input = sample_input.rename(columns=rename_map)

prediction = model.predict(sample_input)
print(f"Sample Prediction for Tip Amount: {prediction[0]:.2f}")

Sample Prediction for Tip Amount: 1.96


In [47]:
experiment =mlflow.get_experiment_by_name("taxi-tip-prediction")
runs =mlflow.search_runs(experiment_ids=[experiment.experiment_id])
print(runs[["run_id", "params.n_estimators","metrics.RMSE", "metrics.MAE", "metrics.R2"]])

                             run_id params.n_estimators  metrics.RMSE  \
0  1ba786630e6e40dab7e5e63767404f8a                None      2.473095   
1  ee4a67872851455e88732d8d9ce922f5                 100      2.408322   
2  618895cdccb740b8b997f3be2ca1aa7c                None      2.320167   
3  3db3207db2f84173be5e7496e643ae0f                 100      2.287291   

   metrics.MAE  metrics.R2  
0     1.241820    0.604794  
1     1.209982    0.625225  
2     1.208029    0.632057  
3     1.185499    0.642410  


In [ ]:
import joblib

# Save model, was done prior and not re-runned
joblib.dump(rf_results, "taxi_model.pkl")

#Part 2: Model Serving with FastAPI

Task 2.3: API Testing

testing- proof of passing

<img src = "test_results.png">


In [ ]:
#!vicorn app:app --reload --port 8000
!pytest test_app.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.1, pytest-9.0.3, pluggy-1.5.0 -- c:\Python313\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\shiva\Downloads\3610 a4
plugins: anyio-4.12.1, flask-1.3.0
collecting ... collected 8 items

test_app.py::test_root PASSED                                            [ 12%]
test_app.py::test_health PASSED                                          [ 25%]
test_app.py::test_predict_valid PASSED                                   [ 37%]
test_app.py::test_predict_missing_field PASSED                           [ 50%]
test_app.py::test_predict_invalid_type PASSED                            [ 62%]
test_app.py::test_predict_out_of_range PASSED                            [ 75%]
test_app.py::test_model_info PASSED                                      [ 87%]
test_app.py::test_batch_prediction PASSED                                [100%]

============================== warnings summary =========

 Verify the auto-generated Swagger UI documentation at /docs is accessible and include
a screenshot

screen shot of swagger ui
 <img src = "swagger_ui_docs.png">

Build the Docker image and report the image size

docker size screenshot
<img src = "docker_image size.png">

In [ ]:
import subprocess, time, requests
subprocess.run(["docker-compose", "up", "--build", "-d"])

url = "http://localhost:8000/predict"

base_payload = {
    "VendorID": 1, "passenger_count": 2, "trip_distance": 30, "RatecodeID": 1, 
    "PULocationID": 161, "DOLocationID": 236, "payment_type": 1, "fare_amount": 12.5, 
    "extra": 0.5, "mta_tax": 0.5, "tolls_amount": 0.0, "improvement_surcharge": 0.3, 
    "congestion_surcharge": 2.5, "Airport_fee": 0.0, "trip_duration_minutes": 15, 
    "trip_speed_mph": 10.0, "pickup_hour": 14, "pickup_day_of_week": 2, 
    "log_trip_distance": 0.916, "fare_per_mile": 5.0, "fare_per_minute": 0.83, 
    "tpep_pickup_datetime_hour": 14, "tpep_pickup_datetime_day": 15, "tpep_pickup_datetime_month": 4, 
    "tpep_dropoff_datetime_hour": 14, "tpep_dropoff_datetime_day": 15, "tpep_dropoff_datetime_month": 4, 
    "store_and_fwd_flag_Y": False, "pickup_Borough_Brooklyn": False, "pickup_Borough_EWR": False, 
    "pickup_Borough_Manhattan": True, "pickup_Borough_N_A": False, "pickup_Borough_Queens": False, 
    "pickup_Borough_Staten_Island": False, "pickup_Borough_Unknown": False, "dropoff_Borough_Brooklyn": False, 
    "dropoff_Borough_EWR": False, "dropoff_Borough_Manhattan": True, "dropoff_Borough_N_A": False, 
    "dropoff_Borough_Queens": False, "dropoff_Borough_Staten_Island": False, "dropoff_Borough_Unknown": False
}
for dist in [10.0, 5.0, 30.0]:
    payload = base_payload.copy()
    payload["trip_distance"] = dist
    res = requests.post(url, json=payload)
    print(f"Dist {dist} miles -> Predicted Tip: {res.json()}")
    
subprocess.run(["docker-compose", "down"])
#chatgpt was used for live pridiction, as testing was dont using the command terminal manually

Dist 10.0 miles -> Predicted Tip: {'prediction': 2.93, 'prediction_id': '8d3bda24-15ec-4942-9123-e820c7e72794', 'model_version': '1.0.0'}
Dist 5.0 miles -> Predicted Tip: {'prediction': 2.93, 'prediction_id': '61330964-0cd8-49b4-b9ab-bba2a9f094cb', 'model_version': '1.0.0'}
Dist 30.0 miles -> Predicted Tip: {'prediction': 2.93, 'prediction_id': 'cee72d7e-3a1e-470d-a7a5-22e12414e03f', 'model_version': '1.0.0'}


CompletedProcess(args=['docker-compose', 'down'], returncode=1)

it seems the model is ignoring the trip distance, and retraining is need but futher analysis is needed for an accurate conclusion. as a k fold eval would need to be preformed. but predict using fast api was working, so there maybe a issue with docker or the way the request is made. ai was used to generate the request code so there is a high change there is an error.